<a href="https://colab.research.google.com/github/marichard/rush_sportswear/blob/additional_insight/GB885_Assignment_7_Richard_M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RUSH | Business Case  
You work as a sales analyst for RUSH, a globally renowned sportswear and footwear brand known for its innovative designs and performance-oriented products. The company stores its raw sales data as a collection of three tables:

*   TABLE_PRODUCTS
*   TABLE_RETAILER  
*   TABLE_SALES

The data includes the number of units sold, the total sales revenue, the location of the sales, the type of product sold, as well as other relevant information. (For data field definitions and explanations, see the data dictionary.) The data is "raw," meaning it has not been cleaned and probably contains errors that need to be addressed.

The VP of US Sales has tasked you with analyzing sales data for trends and insights that will help company leadership understand the market and identify opportunities for growth. For example, you may want to look for trends or insights in seasonality, retailers, locations, or sales methods. Take initiative to apply your creativity and curiosity to this data.

In addition, she has asked you to answer the following business questions:

What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?  
What state had the highest sales (in dollars) of women's products in 2021? How much was it?  
What state had the highest sales (in dollars) of men's products in 2021? How much was it?  
What retailer purchased the most units in 2021? In 2020?

In [135]:
# import libraries
import pandas as pd
import numpy as np

In [136]:
# assign datasets to variables
url1 = 'https://raw.githubusercontent.com/marichard/rush_sportswear/refs/heads/main/TABLE_PRODUCTS_885.csv'
url2 = 'https://raw.githubusercontent.com/marichard/rush_sportswear/refs/heads/main/TABLE_RETAILER_885.csv'
url3 = 'https://raw.githubusercontent.com/marichard/rush_sportswear/refs/heads/main/TABLE_SALES_885.csv'

In [137]:
# create dataframes from variables
products_df = pd.read_csv(url1, sep ='|')
retailer_df = pd.read_csv(url2)
sales_df = pd.read_csv(url3)

# View information of all datasets

In [138]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   PRODUCT_ID    6 non-null      int64 
 1   PRODUCT_NAME  6 non-null      object
dtypes: int64(1), object(1)
memory usage: 228.0+ bytes


In [139]:
products_df.head()

,PRODUCT_ID,PRODUCT_NAME
0,20,Men's Street Footwear
1,30,Men's Athletic Footwear
2,120,Women's Street Footwear
3,130,Women's Athletic Footwear
4,40,Men's Apparel


In [140]:
retailer_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   RETAILER_ID  110 non-null    object
 1   RETAILER     110 non-null    object
 2   REGION       110 non-null    object
 3   STATE        110 non-null    object
 4   CITY         110 non-null    object
dtypes: object(5)
memory usage: 4.4+ KB


In [141]:
retailer_df.head()

,RETAILER_ID,RETAILER,REGION,STATE,CITY
0,A00MOHCO,Amazon,Midwest,Ohio,Columbus
1,A00NMAPO,Amazon,Northeast,Maine,Portland
2,A00NMABO,Amazon,Northeast,Massachusetts,Boston
3,A00NNEMA,Amazon,Northeast,New Hampshire,Manchester
4,A00NVEBU,Amazon,Northeast,Vermont,Burlington


In [142]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDER_ID          9648 non-null   int64  
 1   RETAILER_ID       9648 non-null   object 
 2   INVOICE_DATE      9648 non-null   object 
 3   MONTH             9648 non-null   int64  
 4   DAY               9648 non-null   int64  
 5   YEAR              9648 non-null   int64  
 6   PRODUCT_ID        9648 non-null   int64  
 7   PRICE_PER_UNIT    9646 non-null   float64
 8   UNITS_SOLD        9648 non-null   object 
 9   OPERATING_MARGIN  9648 non-null   float64
 10  SALES_METHOD      9648 non-null   object 
dtypes: float64(2), int64(5), object(4)
memory usage: 829.3+ KB


In [143]:
sales_df.head()

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200,0.5,In-store
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250,0.5,In-store
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220,0.5,Outlet
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200,0.5,Outlet
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220,0.5,Outlet


In [144]:
# check for duplicates of each dataframe
products_duplicates = products_df.duplicated().sum()
retailer_duplicates = retailer_df.duplicated().sum()
sales_duplicates = sales_df.duplicated().sum()
print(f'Number of duplicates in products_df: {products_duplicates}')
print(f'Number of duplicates in retailer_df: {retailer_duplicates}')
print(f'Number of duplicates in sales_df: {sales_duplicates}')

Number of duplicates in products_df: 0
Number of duplicates in retailer_df: 0
Number of duplicates in sales_df: 0


In [145]:
# clean sales dataset
# fix misspelling in of text in sales method column of sales dataset
sales_df['SALES_METHOD'] = sales_df['SALES_METHOD'].replace({'Ootlet': 'Outlet'})

# units sold column
sales_df['UNITS_SOLD'] = sales_df['UNITS_SOLD'].astype(str).str.replace(',', '').str.strip()
sales_df['UNITS_SOLD'] = pd.to_numeric(sales_df['UNITS_SOLD'], errors='coerce')
sales_df['UNITS_SOLD'] = sales_df['UNITS_SOLD'].fillna(sales_df['UNITS_SOLD'].median())

# Fix missing prices using category median
median_price = sales_df[(sales_df['PRODUCT_ID'] == 20) & (sales_df['PRICE_PER_UNIT'] < 1000)]['PRICE_PER_UNIT'].median()
sales_df.loc[sales_df['PRICE_PER_UNIT'] > 1000, 'PRICE_PER_UNIT'] = median_price
sales_df['PRICE_PER_UNIT'] = sales_df['PRICE_PER_UNIT'].fillna(median_price)

# add a total sales column, (price per unit * units sold)
sales_df['TOTAL_SALES'] = sales_df['PRICE_PER_UNIT'] * sales_df['UNITS_SOLD']

In [146]:
# merge datasets
merged_df = pd.merge(sales_df, retailer_df, on='RETAILER_ID')
merged_df = pd.merge(merged_df, products_df, on='PRODUCT_ID')

merged_df.head()

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,TOTAL_SALES,RETAILER,REGION,STATE,CITY,PRODUCT_NAME
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200.0,0.5,In-store,60000.0,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250.0,0.5,In-store,62500.0,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220.0,0.5,Outlet,61000.0,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200.0,0.5,Outlet,60000.0,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220.0,0.5,Outlet,73200.0,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear


In [147]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10270 entries, 0 to 10269
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDER_ID          10270 non-null  int64  
 1   RETAILER_ID       10270 non-null  object 
 2   INVOICE_DATE      10270 non-null  object 
 3   MONTH             10270 non-null  int64  
 4   DAY               10270 non-null  int64  
 5   YEAR              10270 non-null  int64  
 6   PRODUCT_ID        10270 non-null  int64  
 7   PRICE_PER_UNIT    10270 non-null  float64
 8   UNITS_SOLD        10270 non-null  float64
 9   OPERATING_MARGIN  10270 non-null  float64
 10  SALES_METHOD      10270 non-null  object 
 11  TOTAL_SALES       10270 non-null  float64
 12  RETAILER          10270 non-null  object 
 13  REGION            10270 non-null  object 
 14  STATE             10270 non-null  object 
 15  CITY              10270 non-null  object 
 16  PRODUCT_NAME      10270 non-null  object

# What leadership wants to know

In [148]:
# What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
sales_2021 = merged_df[merged_df['YEAR'] == 2021]
product_sales = sales_2021.groupby('PRODUCT_NAME')['TOTAL_SALES'].sum()
top_product_2021 = product_sales.idxmax()
top_product_sales_2021 = product_sales.max()

print(f"The product category with the highest sales in 2021 was {top_product_2021} with a total of ${top_product_sales_2021:,.2f}.")

The product category with the highest sales in 2021 was Men's Street Footwear with a total of $23,294,836.00.


In [149]:
# What state had the highest sales (in dollars) of women's products in 2021? How much was it?
womens_2021 = merged_df[(merged_df['YEAR'] == 2021) & (merged_df['PRODUCT_NAME'].str.contains("Women's", na=False))]
womens_state_sales = womens_2021.groupby('STATE')['TOTAL_SALES'].sum()
womens_top_state_2021 = womens_state_sales.idxmax()
womens_top_state_sales_2021 = womens_state_sales.max()

print(f"The state with the highest sales of women's products in 2021 was {womens_top_state_2021} with a total of ${womens_top_state_sales_2021:,.2f}.")

The state with the highest sales of women's products in 2021 was Maine with a total of $2,176,301.00.


In [150]:
# What state had the highest sales (in dollars) of men's products in 2021? How much was it?
mens_2021 = merged_df[(merged_df['YEAR'] == 2021) & (merged_df['PRODUCT_NAME'].str.contains("Men"))]
mens_state_sales = mens_2021.groupby('STATE')['TOTAL_SALES'].sum()
mens_top_state_2021 = mens_state_sales.idxmax()
mens_top_state_sales_2021 = mens_state_sales.max()

print(f"The state with the highest sales of men's products in 2021 was {mens_top_state_2021} with a total of ${mens_top_state_sales_2021:,.2f}.")

The state with the highest sales of men's products in 2021 was Delaware with a total of $2,334,300.00.


In [151]:
# What retailer purchased the most units in 2021? In 2020?
# 2021
retailer_units_2021 = merged_df[(merged_df['YEAR'] == 2021)].groupby('RETAILER')['UNITS_SOLD'].sum()
top_retailer_2021 = retailer_units_2021.idxmax()
top_retailer_units_2021 = retailer_units_2021.max()

# 2020
retailer_units_2020 = merged_df[(merged_df['YEAR'] == 2020)].groupby('RETAILER')['UNITS_SOLD'].sum()
top_retailer_2020 = retailer_units_2020.idxmax()
top_retailer_units_2020 = retailer_units_2020.max()

print(f"The retailer with the most units purchased in 2021 was {top_retailer_2021} with {top_retailer_units_2021:,.0f} units.")
print(f"The retailer with the most units purchased in 2020 was {top_retailer_2020} with {top_retailer_units_2020:,.0f} units.")

The retailer with the most units purchased in 2021 was Foot Locker with 1,097,410 units.
The retailer with the most units purchased in 2020 was Amazon with 317,930 units.


In [152]:
# results summary
print("=" * 17)
print("EXECUTIVE SUMMARY")
print("=" * 17)
print(f"1. Top Product Category (2021):        {top_product_2021} (${top_product_sales_2021:,.2f})")
print(f"2. Top State for Women's Sales (2021): {womens_top_state_2021} (${womens_top_state_sales_2021:,.2f})")
print(f"3. Top State for Men's Sales (2021):   {mens_top_state_2021} (${mens_top_state_sales_2021:,.2f})")
print(f"4. Top Retailer by Volume (2021):      {top_retailer_2021} ({top_retailer_units_2021:,.0f} units)")
print(f"   Top Retailer by Volume (2020):      {top_retailer_2020} ({top_retailer_units_2020:,.0f} units)")

EXECUTIVE SUMMARY
1. Top Product Category (2021):        Men's Street Footwear ($23,294,836.00)
2. Top State for Women's Sales (2021): Maine ($2,176,301.00)
3. Top State for Men's Sales (2021):   Delaware ($2,334,300.00)
4. Top Retailer by Volume (2021):      Foot Locker (1,097,410 units)
   Top Retailer by Volume (2020):      Amazon (317,930 units)


# Additional Insight

In [154]:
# annual growth and channel contribution trends across sales method's (In-store, Online, and Outlet)
# sales summary
sales_summary = merged_df.groupby(['YEAR', 'SALES_METHOD']).agg(
    Total_Sales=('TOTAL_SALES', 'sum'),
    Total_Units=('UNITS_SOLD', 'sum'),
    Order_Count=('ORDER_ID', 'count')
).reset_index()

sales_summary['Sales_Formatted'] = sales_summary['Total_Sales'].apply(lambda x: f"${x:,.2f}")
sales_summary

,YEAR,SALES_METHOD,Total_Sales,Total_Units,Order_Count,Sales_Formatted
0,2020,In-store,9374550.0,156575.0,287,"$9,374,550.00"
1,2020,Online,4519966.0,87085.0,530,"$4,519,966.00"
2,2020,Outlet,10327059.0,218689.0,485,"$10,327,059.00"
3,2021,In-store,26274075.0,533415.0,1453,"$26,274,075.00"
4,2021,Online,42454662.0,897572.0,4887,"$42,454,662.00"
5,2021,Outlet,29442520.0,639446.0,2628,"$29,442,520.00"
